# Smartphone Addiction: Baseline Modeling

Playground Series S6E8 — Phase 2 of `docs/2_implementation_plan.md`: sanity baselines, strong native-categorical models, an `_is_missing` OOF ablation, EDA-informed engineered features, and a class-imbalance A/B — each gated on evidence from `docs/3_eda_insights.md`, not run as a fixed sweep (per the evidence-gated scope in `docs/archive/4_codex_claude_review_log.md` §13.4).

## 1. Config

In [1]:
import os
import time

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

# Mode flags: one per experiment block, per docs/0_coding_standards.md
# ("one flag per experiment rather than commenting code in/out").
RUN_V1_SANITY = True
RUN_V2_STRONG = True
RUN_V2_MISSING_ABLATION = True
RUN_V3_ENGINEERED = True
RUN_CLASS_WEIGHT_ABLATION = True

pd.set_option("display.max_columns", 50)

## 2. Data Loading

In [2]:
if os.path.exists("/kaggle/input/playground-series-s6e8"):
    DATA_DIR = "/kaggle/input/playground-series-s6e8"
else:
    DATA_DIR = "../data"

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

TARGET = "addicted_label"
NUMERIC_FEATURES = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]
CATEGORICAL_FEATURES = ["gender", "stress_level", "academic_work_impact"]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = train[ALL_FEATURES].copy()
y = train[TARGET].copy()
X_test = test[ALL_FEATURES].copy()

for col in CATEGORICAL_FEATURES:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

print(f"X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")

X: (691369, 12), y: (691369,), X_test: (296302, 12)


## 3. Cross-Validation Helper

In [3]:
results = []  # (name, oof_auc, fold_aucs) for the summary table
oof_store = {}  # name -> oof predictions, for sanity checks / future ensembling

def run_cv(name: str, fit_predict_fold, X_df: pd.DataFrame, y_ser: pd.Series) -> np.ndarray:
    """Run stratified 5-fold CV, print per-fold and overall OOF AUC.

    Args:
        name: label for the results table.
        fit_predict_fold: callable(X_tr, y_tr, X_val) -> val_pred_proba.
        X_df: feature frame.
        y_ser: target series.

    Returns:
        OOF prediction array aligned to X_df's row order.
    """
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X_df))
    fold_aucs = []
    start = time.time()
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_df, y_ser)):
        X_tr, X_val = X_df.iloc[tr_idx], X_df.iloc[val_idx]
        y_tr, y_val = y_ser.iloc[tr_idx], y_ser.iloc[val_idx]
        val_pred = fit_predict_fold(X_tr, y_tr, X_val)
        oof[val_idx] = val_pred
        fold_auc = roc_auc_score(y_val, val_pred)
        fold_aucs.append(fold_auc)
    overall_auc = roc_auc_score(y_ser, oof)
    elapsed = time.time() - start
    print(
        f"{name:40s} OOF AUC={overall_auc:.5f}  "
        f"fold std={np.std(fold_aucs):.5f}  ({elapsed:.0f}s)"
    )
    results.append({
        "name": name,
        "oof_auc": overall_auc,
        "fold_auc_mean": np.mean(fold_aucs),
        "fold_auc_std": np.std(fold_aucs),
        "fold_aucs": fold_aucs,
    })
    oof_store[name] = oof
    return oof

## 4. v1 — Sanity Baselines

Constant predictor, logistic regression, and `HistGradientBoostingClassifier` establish the floor and confirm the evaluation pipeline before any tuning, per `docs/2_implementation_plan.md` Phase 2 step 2.

In [4]:
if RUN_V1_SANITY:
    # Constant predictor: no ranking signal by construction, AUC = 0.5
    # (not computed via roc_auc_score, which requires score variation);
    # recorded directly as the theoretical floor.
    results.append({
        "name": "v1a_constant", "oof_auc": 0.5,
        "fold_auc_mean": 0.5, "fold_auc_std": 0.0, "fold_aucs": [0.5] * N_FOLDS,
    })
    print(f"{'v1a_constant':40s} OOF AUC=0.50000  (theoretical floor, not fit)")

v1a_constant                             OOF AUC=0.50000  (theoretical floor, not fit)


In [5]:
if RUN_V1_SANITY:
    def fit_predict_logreg(X_tr, y_tr, X_val):
        pipe = Pipeline([
            ("prep", ColumnTransformer([
                ("num", Pipeline([
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]), NUMERIC_FEATURES),
                ("cat", Pipeline([
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]), CATEGORICAL_FEATURES),
            ])),
            ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
        ])
        pipe.fit(X_tr, y_tr)
        return pipe.predict_proba(X_val)[:, 1]

    _ = run_cv("v1b_logistic_regression", fit_predict_logreg, X, y)

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

v1b_logistic_regression                  OOF AUC=0.91145  fold std=0.00081  (3s)


/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

In [6]:
if RUN_V1_SANITY:
    def fit_predict_hgb(X_tr, y_tr, X_val):
        model = HistGradientBoostingClassifier(
            random_state=SEED, max_iter=200, categorical_features="from_dtype"
        )
        model.fit(X_tr, y_tr)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v1c_hist_gradient_boosting", fit_predict_hgb, X, y)

v1c_hist_gradient_boosting               OOF AUC=0.95733  fold std=0.00076  (12s)


**Insight:** the constant predictor's AUC=0.5 confirms the floor; logistic regression and HGB's actual OOF AUCs (see the summary table in Section 8) confirm the pipeline is producing genuine ranking signal well above that floor before any tuning — exact numbers in `docs/6_baseline_modeling.md`.

## 5. v2 — Strong Models (Native Categorical)

In [7]:
if RUN_V2_STRONG:
    def fit_predict_lgbm(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1,
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    lgbm_oof = run_cv("v2a_lightgbm_native_cat", fit_predict_lgbm, X, y)

v2a_lightgbm_native_cat                  OOF AUC=0.95480  fold std=0.00068  (11s)


In [8]:
if RUN_V2_STRONG:
    def catboost_ready(df: pd.DataFrame) -> pd.DataFrame:
        # CatBoost's pandas-Categorical cat_features path rejects NaN
        # directly ("cat_features must be integer or string ... NaN
        # values should be converted to string") -- give it an explicit
        # "missing" string category instead, still distinct from every
        # real level, so this is native-missing-handling in spirit even
        # though LightGBM/HGB can take the NaN itself.
        out = df.copy()
        for col in CATEGORICAL_FEATURES:
            out[col] = out[col].astype("object").fillna("missing").astype(str)
        return out

    def fit_predict_catboost(X_tr, y_tr, X_val):
        model = CatBoostClassifier(
            random_seed=SEED, iterations=200, depth=6, learning_rate=0.05,
            cat_features=CATEGORICAL_FEATURES, verbose=False,
        )
        model.fit(catboost_ready(X_tr), y_tr)
        return model.predict_proba(catboost_ready(X_val))[:, 1]

    catboost_oof = run_cv("v2b_catboost_native_cat", fit_predict_catboost, X, y)

v2b_catboost_native_cat                  OOF AUC=0.94190  fold std=0.00063  (70s)


**Insight:** compare against v1's sanity baselines in `docs/6_baseline_modeling.md` — native categorical + native missing-value handling should clear the HGB floor if the tree ensembles are extracting more signal than a single boosting pass on ordinal-ish encodings.

## 6. `_is_missing` Indicator Flags — OOF Ablation

Per `docs/3_eda_insights.md` §4.2/§10: the marginal analysis in EDA found no strong target signal in missingness, but explicitly did not rule out a conditional effect. This is the actual test — LightGBM with vs. without explicit `_is_missing` columns alongside native NaN handling, same model/fold setup as v2a for a clean comparison.

In [9]:
if RUN_V2_MISSING_ABLATION:
    X_with_flags = X.copy()
    for col in ALL_FEATURES:
        X_with_flags[f"{col}_is_missing"] = X[col].isna().astype(int)

    def fit_predict_lgbm_flags(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1,
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v2c_lightgbm_plus_missing_flags", fit_predict_lgbm_flags, X_with_flags, y)

v2c_lightgbm_plus_missing_flags          OOF AUC=0.95480  fold std=0.00068  (13s)


**Insight:** compare `v2a_lightgbm_native_cat` vs. `v2c_lightgbm_plus_missing_flags` OOF AUC in `docs/6_baseline_modeling.md` — this is the direct answer to whether `_is_missing` flags earn their place, not the marginal EDA table.

## 7. v3 — Engineered Features

EDA-informed ratio/residual features among the three strongest predictors (`docs/3_eda_insights.md` §3/§10), computed identically on train and test, target-free (`docs/0_coding_standards.md` leakage rule):

In [10]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add EDA-informed ratio/residual features. Target-free, safe to
    compute once outside the CV loop (docs/0_coding_standards.md)."""
    out = df.copy()
    out["social_to_screen_ratio"] = df["social_media_hours"] / df["daily_screen_time_hours"].replace(0, np.nan)
    out["gaming_to_screen_ratio"] = df["gaming_hours"] / df["daily_screen_time_hours"].replace(0, np.nan)
    out["time_budget_residual"] = 24 - (
        df["sleep_hours"] + df["work_study_hours"] + df["daily_screen_time_hours"]
    )
    out["weekend_escalation"] = df["weekend_screen_time"] - df["daily_screen_time_hours"]
    return out

ENGINEERED_FEATURES = [
    "social_to_screen_ratio", "gaming_to_screen_ratio",
    "time_budget_residual", "weekend_escalation",
]

if RUN_V3_ENGINEERED:
    X_engineered = add_engineered_features(X)

    def fit_predict_lgbm_engineered(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1,
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v3_lightgbm_plus_engineered", fit_predict_lgbm_engineered, X_engineered, y)

v3_lightgbm_plus_engineered              OOF AUC=0.95533  fold std=0.00062  (11s)


**Insight:** compare `v3_lightgbm_plus_engineered` against `v2a_lightgbm_native_cat` — ratios/residuals of features a tree ensemble can already split on nonlinearly are not guaranteed to help; exact delta in `docs/6_baseline_modeling.md`.

## 8. Class-Imbalance A/B

AUC is rank-based, so `class_weight` mainly affects optimizer dynamics rather than the final ranking — tested directly rather than assumed either way, per `docs/2_implementation_plan.md` Phase 2 step 5.

In [11]:
if RUN_CLASS_WEIGHT_ABLATION:
    def fit_predict_lgbm_balanced(X_tr, y_tr, X_val):
        model = LGBMClassifier(
            random_state=SEED, n_estimators=200, num_leaves=31,
            learning_rate=0.05, verbose=-1, class_weight="balanced",
        )
        model.fit(X_tr, y_tr, categorical_feature=CATEGORICAL_FEATURES)
        return model.predict_proba(X_val)[:, 1]

    _ = run_cv("v2d_lightgbm_class_weight_balanced", fit_predict_lgbm_balanced, X, y)

v2d_lightgbm_class_weight_balanced       OOF AUC=0.95463  fold std=0.00058  (11s)


**Insight:** compare against `v2a_lightgbm_native_cat` (unweighted) — expect little to no AUC change either way since AUC only depends on score ranking; exact numbers in `docs/6_baseline_modeling.md`.

## 9. Summary and Candidate Sanity Checks

In [12]:
summary = pd.DataFrame(results)[["name", "oof_auc", "fold_auc_mean", "fold_auc_std"]]
summary = summary.sort_values("oof_auc", ascending=False).reset_index(drop=True)
summary

,name,oof_auc,fold_auc_mean,fold_auc_std
0,v1c_hist_gradient_boosting,0.957333,0.957334,0.000764
1,v3_lightgbm_plus_engineered,0.955329,0.955332,0.000623
2,v2c_lightgbm_plus_missing_flags,0.954804,0.954806,0.000682
3,v2a_lightgbm_native_cat,0.954800,0.954802,0.000675
4,v2d_lightgbm_class_weight_balanced,0.954631,0.954633,0.000584
5,v2b_catboost_native_cat,0.941899,0.941903,0.000631
6,v1b_logistic_regression,0.911447,0.911449,0.000811
7,v1a_constant,0.500000,0.500000,0.000000


In [13]:
def candidate_sanity_checks(name: str, oof_pred: np.ndarray, y_true: pd.Series) -> dict:
    """Replaces the old predicted-rate-vs-70.94% check (which assumed a
    threshold AUC optimization doesn't make) per
    docs/2_implementation_plan.md Phase 2 step 6."""
    finite_in_range = bool(np.all(np.isfinite(oof_pred)) and np.all((oof_pred >= 0) & (oof_pred <= 1)))
    n_unique = int(pd.Series(oof_pred).nunique())
    overall_auc = roc_auc_score(y_true, oof_pred)
    return {
        "name": name,
        "finite_in_[0,1]": finite_in_range,
        "n_unique_predictions": n_unique,
        "prediction_range": (float(oof_pred.min()), float(oof_pred.max())),
        "overall_oof_auc": overall_auc,
    }

best_name = summary.iloc[0]["name"] if summary.iloc[0]["name"] != "v1a_constant" else summary.iloc[1]["name"]
checks = candidate_sanity_checks(best_name, oof_store[best_name], y)
pd.Series(checks)

name                       v1c_hist_gradient_boosting
finite_in_[0,1]                                  True
n_unique_predictions                           686565
prediction_range        (1.7233597543566132e-79, 1.0)
overall_oof_auc                              0.957333
dtype: object

**Insight:** the leading candidate (excluding the constant floor) passes basic sanity (finite, bounded, non-degenerate predictions) before being considered for Phase 3 promotion-gate comparison. Full progression table and interpretation in `docs/6_baseline_modeling.md`.

## 10. Next Moves

Feeds Phase 3 (`docs/2_implementation_plan.md`): the strongest v1–v3 configuration here becomes the champion baseline for the evidence-gated tuning/ensemble sequence — hand-designed parameter search first, Optuna only if headroom justifies it, ensemble only with paired evidence. Promotion-gate thresholds for Phase 3 will be derived from this notebook's fold-to-fold OOF AUC std (Section 9), not a borrowed number.